In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv('csv/train.csv')

for col in df.select_dtypes(include=['object', 'string', 'str']).columns:
    df[col] = df[col].str.strip()

df.replace('NaN', np.nan)
df.dropna(subset=['Time_Orderd', 'Time_Order_picked', 'Time_taken(min)'], inplace=True)

df['Time_taken(min)'] = df['Time_taken(min)'].str.extract(r'(\d+)').astype(float)

df['Order_Date'] = pd.to_datetime(df['Order_Date'], format="%d-%m-%Y")

df['Time_Orderd'] = df['Time_Orderd'].apply(lambda x: str(x) + ':00' if len(str(x)) <= 5 else str(x))
df['Time_Order_picked'] = df['Time_Order_picked'].apply(lambda x: str(x) + ':00' if len(str(x)) <= 5 else str(x))

df['Time_Orderd'] = pd.to_timedelta(df['Time_Orderd'], errors='coerce')
df['Time_Order_picked'] = pd.to_timedelta(df['Time_Order_picked'], errors='coerce')

df.dropna(subset=['Time_Orderd', 'Time_Order_picked'], inplace=True)

time_diff = (df['Time_Order_picked'] - df['Time_Orderd']).dt.total_seconds() / 60

df['prep_time'] = np.where(time_diff < 0, time_diff + 1440, time_diff)
df['travel_time'] = df['Time_taken(min)'] - df['prep_time']

df = df[(df['prep_time'] > 0) & (df['prep_time'] < 60)]
df = df[(df['travel_time'] > 0) & (df['travel_time'] < 120)]

print(df[['Time_Orderd', 'Time_Order_picked', 'prep_time', 'travel_time', 'Time_taken(min)']].head())

      Time_Orderd Time_Order_picked  prep_time  travel_time  Time_taken(min)
0 0 days 11:30:00   0 days 11:45:00       15.0          9.0             24.0
1 0 days 19:45:00   0 days 19:50:00        5.0         28.0             33.0
2 0 days 08:30:00   0 days 08:45:00       15.0         11.0             26.0
3 0 days 18:00:00   0 days 18:10:00       10.0         11.0             21.0
4 0 days 13:30:00   0 days 13:45:00       15.0         15.0             30.0
